# SMM Final Project — Colab GPU Runner

跨國 zero-shot 實驗的 Colab 執行簿本。用 GPU 跑 TSET / DMC / AMCv2。

**使用前提**：把整個專案資料夾（含 `src/` 與 `data/processed/`）上傳到 Google Drive。
資料 ~3.6GB，放 Drive 一次、長期重用，不要每次 upload。

**建議畫面設定**：上方選單 Runtime → Change runtime type → Hardware accelerator 選 **T4 GPU**（或更好）。

## 1. 確認 GPU 已啟用

In [ ]:
!nvidia-smi

## 2. 挂載 Google Drive

授權後，你的 Drive 會出現在 `/content/drive/MyDrive/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 設定專案路徑

把 `PROJECT_DIR` 改成你 Drive 裡放專案的位置。該路徑底下要有 `src/` 與 `data/processed/`。

In [ ]:
import os

# ← 改成你的實際路徑
PROJECT_DIR = '/content/drive/MyDrive/SMM_Final_Project'

assert os.path.isdir(os.path.join(PROJECT_DIR, 'src')), f'找不到 src/，檢查 PROJECT_DIR: {PROJECT_DIR}'
assert os.path.isdir(os.path.join(PROJECT_DIR, 'data', 'processed')), '找不到 data/processed/，請確認資料已上傳'
print('專案路徑 OK')
print('國家資料夾:', sorted(os.listdir(os.path.join(PROJECT_DIR, 'data', 'processed'))))

### （選用）把 code 複製到本地加速

在 Drive 上直接讀寫 `data/interim/` 會比較慢（實驗會寫 model、圖、metrics）。
這裡把 `src/` 複到 Colab 本地磁碟，並用 symlink 把 `data/` 指回 Drive（資料太大不複，只讀）。
interim 輸出會寫在本地，跨國結果 log 仍會回寫 Drive 的 `results/`。

In [ ]:
import shutil, pathlib

LOCAL_DIR = '/content/smm'
os.makedirs(LOCAL_DIR, exist_ok=True)

# 複 src/ 到本地
if os.path.exists(f'{LOCAL_DIR}/src'):
    shutil.rmtree(f'{LOCAL_DIR}/src')
shutil.copytree(f'{PROJECT_DIR}/src', f'{LOCAL_DIR}/src')

# data/ 太大，用 symlink 指回 Drive（只讀）
if os.path.islink(f'{LOCAL_DIR}/data') or os.path.exists(f'{LOCAL_DIR}/data'):
    os.remove(f'{LOCAL_DIR}/data') if os.path.islink(f'{LOCAL_DIR}/data') else shutil.rmtree(f'{LOCAL_DIR}/data')
os.symlink(f'{PROJECT_DIR}/data', f'{LOCAL_DIR}/data')

# results/ 直接用 Drive 的（log 要保留）
if os.path.islink(f'{LOCAL_DIR}/results') or os.path.exists(f'{LOCAL_DIR}/results'):
    os.remove(f'{LOCAL_DIR}/results') if os.path.islink(f'{LOCAL_DIR}/results') else shutil.rmtree(f'{LOCAL_DIR}/results')
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.symlink(f'{PROJECT_DIR}/results', f'{LOCAL_DIR}/results')

RUN_DIR = LOCAL_DIR  # 之後用這個當 project root
print('本地 RUN_DIR:', RUN_DIR)
print(os.listdir(RUN_DIR))

## 4. 安裝依賴

Colab 已預裝 torch（帶 CUDA）。這裡裝 torch_geometric 與專案用到的其他套件。
torch_geometric 的 GCN/SAGE/GAT 在新版不需要編譯版 torch_scatter。

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

# 只裝 torch_geometric 與 mlflow。
# 不要裝 node2vec：它會拉進 gensim 把 numpy 降到 <2，與 Colab 的 numpy 2.x
# 套件 binary 不相容（numpy.dtype size changed）。TSET/DMC/AMCv2 不需要 node2vec，
# my_utils.py 已把該 import 改成 lazy（只有跑 Node2Vec baseline 才會用到）。
!pip install -q torch_geometric mlflow
# networkx / scikit-learn / scipy / pandas Colab 已有，不用裝

import torch_geometric, mlflow, networkx, sklearn
print('torch_geometric', torch_geometric.__version__, '— 依賴 OK')

## 5. Smoke test：先單跑一國確認 GPU pipeline 通

`--device 0` 會自動使用 GPU（`setup_env` 偵測 `cuda.is_available()`）。
先跑 iran（資料較小）驗證。

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method tset --countries iran --device 0

## 6. 正式批次：六國 × 三方法

一次跑一個方法，結果即時寫回 Drive 的 `results/`（即使 session 斷線也不會遺失已跑完的國家）。
免費版可能撃不完三個方法，建議分個 cell 跑。

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method tset --device 0

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method dmc --device 0

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method amcv2 --device 0

## 7. 確認結果

Log 已寫在 Drive 的 `results/`。此處快速抽出每國的最終 test f1_macro。

In [ ]:
import glob, re

results_dir = f'{PROJECT_DIR}/results'
for f in sorted(glob.glob(f'{results_dir}/zero-shot_TSET_*.txt')):
    country = re.search(r'TSET_(\w+)\.txt', f).group(1)
    txt = open(f, encoding='utf-8', errors='replace').read()
    sotm = re.search(r'Structure-Only Mode \(SOTM\)\s+=\s+(\w+)', txt)
    mmd = re.search(r'min_MMD.*?=\s+([\d.]+)', txt)
    # 最後一個 TEST f1_macro
    f1s = re.findall(r'TEST.*?f1_macro[^0-9]*([\d.]+)', txt)
    print(f'{country:12s} SOTM={sotm.group(1) if sotm else "?":6s} '
          f'min_MMD={mmd.group(1) if mmd else "?":8s} '
          f'f1_macro={f1s[-1] if f1s else "(未完成)"}')